In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from etf_momentum_backtest.config import BacktestConfig
from etf_momentum_backtest.data import load_prices
from etf_momentum_backtest.experiments import (
    ParameterGrid,
    build_parameter_pivot,
    rank_experiments,
    run_parameter_grid,
    save_experiment_results,
    select_robust_candidates,
)


In [ ]:
base_config = BacktestConfig(
    tickers=("SPY", "QQQ", "TLT", "IEF", "GLD"),
    start_date="2005-01-01",
    end_date=None,
    lookback_days=126,
    top_k=2,
    transaction_cost_rate=0.001,
    initial_capital=1_000_000.0,
)

base_config


In [ ]:
prices = load_prices(
    config=base_config,
    refresh=False,
)

print(prices.index.min(), prices.index.max())
prices.tail()


In [ ]:
grid = ParameterGrid(
    lookback_days=(63, 126, 189, 252),
    top_k=(1, 2, 3),
    transaction_cost_bps=(0, 10, 25),
)

len(grid.combinations(len(base_config.tickers)))


In [ ]:
results = run_parameter_grid(
    prices=prices,
    base_config=base_config,
    grid=grid,
)

results


In [ ]:
results.loc[
    results["status"].ne("ok"),
    ["experiment_id", "error"],
]


In [ ]:
ranked = rank_experiments(
    results,
    metric="sharpe_ratio",
)

ranked[
    [
        "rank",
        "experiment_id",
        "lookback_days",
        "top_k",
        "transaction_cost_bps",
        "cagr",
        "sharpe_ratio",
        "max_drawdown",
        "annualized_turnover",
    ]
].head(10)


In [ ]:
candidates = select_robust_candidates(
    results,
    min_sharpe=0.60,
    max_drawdown_limit=-0.35,
    max_annualized_turnover=6.0,
)

candidates.sort_values(
    "sharpe_ratio",
    ascending=False,
).head(10)


In [ ]:
sharpe_pivot = build_parameter_pivot(
    results,
    metric="sharpe_ratio",
    transaction_cost_bps=10,
)

sharpe_pivot


In [ ]:
figure, axis = plt.subplots(figsize=(9, 5))

for top_k in sharpe_pivot.columns:
    axis.plot(
        sharpe_pivot.index,
        sharpe_pivot[top_k],
        marker="o",
        label=f"Top {top_k}",
    )

axis.set_title("Sharpe Ratio Parameter Sensitivity")
axis.set_xlabel("Lookback Days")
axis.set_ylabel("Sharpe Ratio")
axis.grid(alpha=0.3)
axis.legend()
figure.tight_layout()
plt.show()


In [ ]:
output_path = Path(
    "outputs/experiments/momentum_parameter_sweep.csv"
)

save_experiment_results(
    results=results,
    output_path=output_path,
)

output_path
